In [110]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [111]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [112]:
len(words)

32033

In [113]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [114]:
# let's build the dataset
block_size = 3 #context length
dim = 10 #the number of dimensions we are choosing to explain the behavior of the data

def build_dataset(words):
    X, Y = [], []
    
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            X.append(context)
            ix = stoi[ch]
            Y.append(ix)
            context = context[1:] + [ix]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

X,Y = build_dataset(words)
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [115]:
# utility function
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff:{maxdiff}')

In [116]:
n_embeddings = 10
n_hiddenneurons = 64

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embeddings), generator = g)

# Layer 1
W1 = torch.randn((n_embeddings * block_size, n_hiddenneurons), generator = g) * (5/3)/((n_embeddings * block_size) ** 0.5)
b1 = torch.randn(n_hiddenneurons, generator = g) * 0.1

# Layer 2
W2 = torch.randn((n_hiddenneurons, vocab_size), generator = g) * 0.1
b2 = torch.randn(vocab_size, generator = g) * 0.1

# batch norm stuff
bngain = torch.randn((1, n_hiddenneurons))*0.1 + 1.0
bnbias = torch.randn((1, n_hiddenneurons))*0.1

# all parameters
parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True


4137


In [117]:
batch_size = 32
n = batch_size
ix = torch.randint(0, Xtr.shape[0], (batch_size, ), generator = g)
Xb, Yb = Xtr[ix], Ytr[ix]

In [118]:
# forward pass - every step written explicitly so that backward prop can be done easily

emb = C[Xb] # get the embeddings of the input vectors
embcat = emb.view(emb.shape[0], -1) # concatenating the input vectors

# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer 1 pre batch normalisation and activation

# batch norm layer
bndiff = hprebn - hprebn.sum(0, keepdim = True)/n
bndiff2 = bndiff**2
bnvar = bndiff2.sum(0, keepdim = True)/(n-1) # n-1 because bessel's correction while calculating sample mean
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv # gaussian thingy - normalised perfectly
hpreact = bngain * bnraw + bnbias # hidden layer 1 pre batch normalisation and activation | bngain is like w and bnbias is like b - nothing fancy

# non linearity - let's use tanh here
h = torch.tanh(hpreact) # hidden layer

# Linear layer 2
logits = h @ W2 + b2 # output layer

# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# PyTorch backward pass
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss



tensor(3.3271, grad_fn=<NegBackward0>)

In [174]:
emb.shape, C.shape, Xb.shape

(torch.Size([32, 3, 10]), torch.Size([27, 10]), torch.Size([32, 3]))

In [202]:
demb.view(-1, demb.shape[-1])

tensor([[ 2.6211e-05, -1.6652e-03,  1.3658e-04,  7.9745e-04, -1.4472e-03,
         -2.7742e-03,  4.0988e-03,  3.7266e-04, -1.5296e-04,  1.9994e-03],
        [-2.5253e-03,  1.7197e-03,  1.2519e-03,  3.4788e-03, -2.5198e-04,
          5.2132e-04,  4.0007e-04,  2.9713e-03,  3.4615e-03, -4.0916e-03],
        [ 2.9758e-03,  4.0384e-03,  7.5165e-05,  5.1586e-03, -5.5462e-03,
          1.2077e-03,  4.4879e-03,  1.1356e-03,  3.6229e-03, -2.3289e-03],
        [-1.8818e-03, -3.7138e-03,  1.2079e-05,  7.9257e-03,  1.7372e-03,
         -5.3993e-03,  3.5652e-03, -1.9499e-03, -2.8720e-03,  1.8654e-03],
        [-1.6135e-03, -1.5577e-03, -9.2932e-04, -1.8076e-03,  2.4969e-03,
         -4.4097e-03,  5.9606e-04, -2.3886e-03,  1.9633e-03,  9.7586e-04],
        [ 1.7839e-03, -2.7365e-03,  1.9495e-04,  1.0394e-03, -5.6949e-03,
          5.7659e-04, -2.0120e-03,  9.2609e-05, -2.0247e-03,  7.4203e-03],
        [-5.5343e-04, -4.7800e-03,  4.4306e-03,  5.7140e-03,  2.6602e-03,
          6.0974e-03,  3.9292e-0

In [208]:
# Exercise 1: backprop through the whole thing manually, 
# backpropagating through exactly all of the variables 
# as they are defined in the forward pass above, one by one

#dl/dlogprobs = -1/n for everything who participate and 0 for the rest

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0/n
dprobs = 1/probs * dlogprobs
inter = counts * dprobs
dcounts_sum_inv = inter[range(n), Yb].view(-1, 1)
dcounts_sum = -1*(counts_sum**-2) * dcounts_sum_inv
dcounts = counts_sum_inv * dprobs
dcounts +=  torch.ones_like(counts) * dcounts_sum
dnorm_logits = counts * dcounts
dlogit_maxes = (-dnorm_logits).sum(1, keepdim = True)
dlogits = dnorm_logits + F.one_hot(logits.max(1).indices, num_classes = logits.shape[1]) * dlogit_maxes

# Linear layer 2
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)

dhpreact = (1-h**2) * dh
dbngain = (bnraw * dhpreact).sum(0, keepdim = True)
dbnbias = dhpreact.sum(0, keepdim = True)
dbnraw = bngain * dhpreact
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim = True)
dbnvar = -0.5 * ((bnvar + 1e-5)**-1.5) * dbnvar_inv
dbndiff2 = torch.ones_like(bndiff2) * dbnvar/(n-1)
dbndiff = bnvar_inv * dbnraw + 2*bndiff * dbndiff2
dhprebn = dbndiff - (1/n) * dbndiff.sum(0) * torch.ones_like(hprebn)

# Linear layer 1
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)

demb = dembcat.view(emb.shape)
demb[Xb]

dC = torch.zeros_like(C)
for i in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix = Xb[i,j]
        dC[ix] += demb[i,j]
        
cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
# cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
#cmp('C', dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff:0.0
probs           | exact: True  | approximate: True  | maxdiff:0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff:0.0
counts_sum      | exact: True  | approximate: True  | maxdiff:0.0
counts          | exact: True  | approximate: True  | maxdiff:0.0
norm_logits     | exact: True  | approximate: True  | maxdiff:0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff:0.0
logits          | exact: True  | approximate: True  | maxdiff:0.0
h               | exact: True  | approximate: True  | maxdiff:0.0
W2              | exact: True  | approximate: True  | maxdiff:0.0
b2              | exact: True  | approximate: True  | maxdiff:0.0
hpreact         | exact: True  | approximate: True  | maxdiff:0.0
bngain          | exact: True  | approximate: True  | maxdiff:0.0
bnbias          | exact: True  | approximate: True  | maxdiff:0.0
bnraw           | exact: True  | approximate: True  | maxdiff:0.0
bnvar_inv 

In [209]:
# Exercise 2: backprop through cross_entropy but all in one go
# to complete this challenge look at the mathematical expression of the loss,
# take the derivative, simplify the expression, and just write it out

# forward pass

# before:
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# now:
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())

3.3270881175994873 diff: 2.384185791015625e-07


In [218]:
logits.shape, Yb.shape, logit_maxes.shape, logprobs.shape, loss.shape

(torch.Size([32, 27]),
 torch.Size([32]),
 torch.Size([32, 1]),
 torch.Size([32, 27]),
 torch.Size([]))

In [232]:
# backward pass

# dlogits = (1/n)*(probs - F.one_hot(Yb, num_classes = probs.shape[1]))
dlogits = (1/n)*(F.softmax(logits, 1) - F.one_hot(Yb, num_classes = probs.shape[1]))

cmp('logits', dlogits, logits) # I can only get approximate to be true, my maxdiff is 6e-9

logits          | exact: False | approximate: True  | maxdiff:4.6566128730773926e-09


In [233]:
# Exercise 3: backprop through batchnorm but all in one go
# to complete this challenge look at the mathematical expression of the output of batchnorm,
# take the derivative w.r.t. its input, simplify the expression, and just write it out
# BatchNorm paper: https://arxiv.org/abs/1502.03167

# forward pass

# before:
# bnmeani = 1/n*hprebn.sum(0, keepdim=True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff**2
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
# bnvar_inv = (bnvar + 1e-5)**-0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw + bnbias

# now:
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias
print('max diff:', (hpreact_fast - hpreact).abs().max())

max diff: tensor(4.7684e-07, grad_fn=<MaxBackward1>)


In [ ]:
# backward pass

# before we had:
# dbnraw = bngain * dhpreact
# dbndiff = bnvar_inv * dbnraw
# dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
# dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
# dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
# dbndiff += (2*bndiff) * dbndiff2
# dhprebn = dbndiff.clone()
# dbnmeani = (-dbndiff).sum(0)
# dhprebn += 1.0/n * (torch.ones_like(hprebn) * dbnmeani)

# calculate dhprebn given dhpreact (i.e. backprop through the batchnorm)
# (you'll also need to use some of the variables from the forward pass up above)

# -----------------
# YOUR CODE HERE :)
fx = bndiff
gx = bnvar**0.5
f1x = 
dhprebn = None # TODO. my solution is 1 (long) line
# -----------------

cmp('hprebn', dhprebn, hprebn) # I can only get approximate to be true, my maxdiff is 9e-10

In [234]:
hprebn.shape, hpreact.shape, bngain.shape, bnbias.shape

(torch.Size([32, 64]),
 torch.Size([32, 64]),
 torch.Size([1, 64]),
 torch.Size([1, 64]))